# 作业三：文档隐含义/主题挖掘与推荐系统

基于 **LSA/LSI/LDA 降维** + **KNN 推荐** + **PCA/t-SNE/UMAP 可视化**。

数据：作业一 `ai_corpus`、作业二混合语料；对照实验使用 `20 Newsgroups` 子集。

## 0. 理论背景

**LSA/LSI** 基于 SVD 对词项-文档矩阵降维，优势是高效、简单、可解释；劣势是对一词多义不敏感、新文档需重新投影。

**LDA** 假设文档由若干隐主题混合生成，用 EM 估计主题分布。

**UMAP** 假设数据分布在黎曼流形上，通过局部邻域优化将高维嵌入映射到二维。

本实现使用 `sklearn + numpy`（Gensim 在 Python 3.14 暂无预编译包）。

In [1]:
import os, sys
from pathlib import Path

cwd = Path.cwd().resolve()
HW3 = cwd if cwd.name == 'homewprk_3' else cwd / 'homewprk_3'
if not (HW3 / 'main.py').is_file():
    raise RuntimeError('请在仓库根目录或 homewprk_3 目录下打开本 Notebook')
sys.path.insert(0, str(HW3))
os.chdir(HW3)
print('工作目录:', os.getcwd())

工作目录: D:\vscode\dataCollection\homewprk_3


## 1. 数据加载

In [5]:
from data_loader import load_chinese_ai_corpus, load_20newsgroups_subset

zh_docs, zh_meta = load_chinese_ai_corpus()
en_docs, en_meta = load_20newsgroups_subset()
print('中文文档数:', len(zh_docs))
print('英文文档数:', len(en_docs))
print('\n中文样例:', zh_docs[0][:100], '...')

中文文档数: 75
英文文档数: 120

中文样例: 人工智能（Artificial Intelligence，简称AI）是计算机科学的一个分支，旨在研究和开发能够模拟、延伸和扩展人类智能的理论、方法、技术及应用系统。 ...


## 2. 分词与 TF-IDF / BM25

In [ ]:
from preprocess import tokenize_docs, build_tfidf, build_count, bm25_scores

zh_tok = tokenize_docs(zh_docs, lang='zh')
tfidf, tfidf_vec = build_tfidf(zh_tok)
count, count_vec = build_count(zh_tok)
bm25 = bm25_scores(tfidf)
print('TF-IDF 形状:', tfidf.shape)
print('词表样例:', list(tfidf_vec.get_feature_names_out()[:15]))

## 3. SVD 理论演示（LSA/LSI 基础）

对文档-词项矩阵做 SVD：$A = U \Sigma V^T$，取前 $k$ 个奇异值得到低秩逼近。

In [ ]:
import numpy as np
from topic_models import numpy_svd_demo
from visualize import plot_singular_values
from config import OUTPUT_DIR, LSA_COMPONENTS

dense = tfidf.toarray()
svd_info = numpy_svd_demo(dense, k=LSA_COMPONENTS)
print('前', LSA_COMPONENTS, '维能量占比:', f"{svd_info['top_k_energy_ratio']:.2%}")
plot_singular_values(svd_info['singular_values'], os.path.join(OUTPUT_DIR, 'notebook_svd.png'))

## 4b. BM25 风格检索（补充 TF-IDF）

In [ ]:
from recommender import query_by_keywords

print('BM25/关键词检索 top5:')
for h in query_by_keywords(tfidf, tfidf_vec, ['深度学习', '神经网络', '大模型']):
    print(f"  #{h['doc_id']} score={h['score']:.4f}")
    print(' ', zh_docs[h['doc_id']][:90], '...')

## 4. LSI 与 LDA 主题模型

In [ ]:
from topic_models import train_lsi, train_lda, top_terms_per_topic
from visualize import plot_topic_terms
from config import N_TOPICS, LDA_MAX_ITER, RANDOM_STATE

lsi_model, lsi_topics = train_lsi(tfidf, n_components=LSA_COMPONENTS, random_state=RANDOM_STATE)
lda_model, lda_topics = train_lda(count, n_topics=N_TOPICS, max_iter=LDA_MAX_ITER, random_state=RANDOM_STATE)

lsi_terms = top_terms_per_topic(lsi_model, tfidf_vec.get_feature_names_out(), kind='lsi')
for t in lsi_terms:
    words = ', '.join(w for w, _ in t['terms'][:6])
    print(f"LSI Topic {t['topic_id']}: {words}")

plot_topic_terms(lsi_terms, os.path.join(OUTPUT_DIR, 'notebook_lsi_topics.png'), n_topics=LSA_COMPONENTS)
plot_topic_terms(
    top_terms_per_topic(lda_model, count_vec.get_feature_names_out(), kind='lda'),
    os.path.join(OUTPUT_DIR, 'notebook_lda_topics.png'),
    n_topics=N_TOPICS,
)

## 5. 文档嵌入可视化（PCA / t-SNE / UMAP）

In [ ]:
from topic_models import assign_dominant_topic
from visualize import reduce_2d, plot_embedding_scatter
from config import UMAP_NEIGHBORS, UMAP_MIN_DIST

labels = [f"T{assign_dominant_topic(lsi_topics)[i]}" for i in range(len(zh_docs))]
for method in ('pca', 'tsne', 'umap'):
    coords = reduce_2d(lsi_topics, method=method, umap_neighbors=UMAP_NEIGHBORS, umap_min_dist=UMAP_MIN_DIST)
    plot_embedding_scatter(coords, labels, f'中文语料 LSI — {method.upper()}', os.path.join(OUTPUT_DIR, f'notebook_{method}.png'))
    print(method, '完成')

## 6. KNN 文档推荐

## 8. 结果图表展示

In [ ]:
from IPython.display import Image, display
import json

for name in [
    'chinese_ai_svd_spectrum.png', 'chinese_ai_lsi_topics.png', 'chinese_ai_lda_topics.png',
    'chinese_ai_embed_pca.png', 'chinese_ai_embed_tsne.png', 'chinese_ai_embed_umap.png',
    'newsgroups_embed_umap.png',
]:
    p = os.path.join(OUTPUT_DIR, name)
    if os.path.isfile(p):
        print(name)
        display(Image(filename=p))

with open(os.path.join(OUTPUT_DIR, 'summary.json'), encoding='utf-8') as f:
    print(json.load(f))

In [ ]:
from recommender import build_knn_index, recommend, query_by_keywords
from config import KNN_K

nn = build_knn_index(lsi_topics, metric='cosine', n_neighbors=KNN_K + 1)
q = 0
recs = recommend(nn, lsi_topics, q, k=KNN_K)
print('查询文档:', zh_docs[q][:120], '...\n')
for r in recs:
    print(f"推荐 #{r['doc_id']} 距离={r['distance']:.4f}")
    print(' ', zh_docs[r['doc_id']][:100], '...\n')

print('关键词检索（人工智能, 模型）:')
for h in query_by_keywords(tfidf, tfidf_vec, ['人工智能', '模型']):
    print(f"  #{h['doc_id']} score={h['score']:.4f} | {zh_docs[h['doc_id']][:80]}...")

## 7. 一键运行完整流水线（含英文对照集）

In [ ]:
from main import main
main()